Feature selection using Mutual info Classification

Note: always make sure that your model is being trained by features that impacts the target feature.
linear relatioships are not the only important consideration.

if feature mutal score for a column is 0.0, it means that feature doesnt impact the target feature. 

In [1]:
# Loading our datas
import pandas as pd
import numpy as np

data_files = {
    'sessions': ('session_time',),
    'riders': ('signup_date',),
    'trips': ('pickup_time', 'dropoff_time'),
    'promotions': ('start_date', 'end_date'),
    'drivers': ('signup_date', 'last_active')
}

dfs={}
for table, date_cols in data_files.items():
    dfs[table] = pd.read_csv(f'../data/{table}.csv', parse_dates=list(date_cols))

In [3]:
print(dfs['promotions'].to_string())

   promo_id           promo_name    promo_type  promo_value start_date   end_date target_segment  city_scope                         ab_test_groups  test_allocation   success_metric
0      P000       Peak Hour Pass  surge_waiver          1.0 2025-04-26 2025-05-25            All     Nairobi                                ['All']            [1.0]  Usage Frequency
1      P001       Peak Hour Pass  surge_waiver          1.0 2025-04-26 2025-05-22            All       Cairo  ['Control', 'Variant A', 'Variant B']  [0.3, 0.4, 0.3]  Conversion Rate
2      P002       Peak Hour Pass  surge_waiver          1.0 2025-04-26 2025-05-16            All       Cairo  ['Control', 'Variant A', 'Variant B']  [0.3, 0.4, 0.3]              ROI
3      P003        Loyalty Bonus        points        100.0 2025-04-26 2025-05-04          Gold+     Nairobi  ['Control', 'Variant A', 'Variant B']  [0.3, 0.4, 0.3]  Conversion Rate
4      P004        Loyalty Bonus        points        100.0 2025-04-26 2025-05-15         

In [2]:
# Merging our dataset
trips = dfs['trips']
riders = dfs['riders']
sessions = dfs['sessions']
drivers = dfs['drivers']
promotions = dfs['promotions']

# We realise a naming mismatch in the User_id in trips and sessions
sessions.rename(columns={'rider_id':'user_id'}, inplace=True)

sessions



,session_id,user_id,session_time,time_on_app,pages_visited,converted,city,loyalty_status
0,S000000,R08605,2025-04-27 18:57:06+02:05,79,4,1,Cairo,Bronze
1,S000001,R08823,2025-04-27 07:32:22+02:27,101,3,0,Nairobi,Silver
2,S000002,R05342,2025-04-27 23:17:25+02:05,12,1,0,Cairo,Bronze
3,S000003,R05057,2025-04-27 14:40:25+00:14,19,1,0,Lagos,Silver
4,S000004,R09614,2025-04-27 08:31:22+00:14,4,1,0,Lagos,Bronze
...,...,...,...,...,...,...,...,...
49995,S049995,R03092,2025-04-27 07:52:18+00:14,76,5,0,Lagos,Bronze
49996,S049996,R02272,2025-04-27 16:52:12+02:05,44,1,1,Cairo,Bronze
49997,S049997,R06892,2025-04-27 14:02:26+02:27,9,5,0,Nairobi,Bronze
49998,S049998,R03121,2025-04-27 03:45:34+00:14,65,2,0,Lagos,Bronze


In [3]:
import pandas as pd

#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)

merged_DF = (
    trips.merge(riders, on='user_id', how='left')
    .merge(sessions, on='user_id', how='left')
    .merge(drivers, on='driver_id', how='left', suffixes=('_driver', '_rider'))
)

merged_DF


,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,...,pages_visited,converted,city_driver,loyalty_status,rating,vehicle_type,signup_date_rider,last_active,city_rider,acceptance_rate
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,...,1.0,0.0,Nairobi,Bronze,4.1,Sedan,2024-07-21,2025-03-21 22:47:26.558938,Nairobi,0.549628
1,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,...,1.0,0.0,Nairobi,Bronze,4.1,Sedan,2024-07-21,2025-03-21 22:47:26.558938,Nairobi,0.549628
2,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,...,1.0,0.0,Nairobi,Bronze,4.1,Sedan,2024-07-21,2025-03-21 22:47:26.558938,Nairobi,0.549628
3,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,...,1.0,0.0,Nairobi,Bronze,4.1,Sedan,2024-07-21,2025-03-21 22:47:26.558938,Nairobi,0.549628
4,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 23:13:48+00:14,2024-10-28 23:26:48+00:14,6.675266,...,5.0,0.0,Lagos,Gold,4.9,SUV,2023-05-06,2025-04-12 08:36:21.207528,Lagos,0.629250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1001269,T199998,R02867,D00974,17.18,1.3,0.00,Mobile Money,2024-09-25 03:11:33+00:14,2024-09-25 03:45:33+00:14,6.540074,...,4.0,0.0,Lagos,Gold,3.5,Sedan,2023-09-26,2025-03-28 06:20:43.379153,Lagos,0.579188
1001270,T199999,R07749,D04894,13.47,1.0,0.00,Card,2024-05-24 18:19:39+02:05,2024-05-24 18:51:39+02:05,30.234277,...,1.0,0.0,Cairo,Bronze,5.0,Luxury,2023-03-30,2025-04-04 14:52:39.161225,Cairo,0.789828
1001271,T199999,R07749,D04894,13.47,1.0,0.00,Card,2024-05-24 18:19:39+02:05,2024-05-24 18:51:39+02:05,30.234277,...,2.0,0.0,Cairo,Bronze,5.0,Luxury,2023-03-30,2025-04-04 14:52:39.161225,Cairo,0.789828
1001272,T199999,R07749,D04894,13.47,1.0,0.00,Card,2024-05-24 18:19:39+02:05,2024-05-24 18:51:39+02:05,30.234277,...,1.0,1.0,Cairo,Bronze,5.0,Luxury,2023-03-30,2025-04-04 14:52:39.161225,Cairo,0.789828


To check the correlationbetween dependent and independent variables

In [4]:
# We first select the columns to deal with
numeric_cols = merged_DF.select_dtypes(include='float64').columns.tolist()
churn_correlation = merged_DF[numeric_cols].corr()['churn_prob']

churn_correlation

# We see that most almost all features has no linear relationship with the target variable 

fare                0.000277
surge_multiplier    0.000160
tip                -0.001150
pickup_lat         -0.000634
pickup_lng          0.009449
dropoff_lat        -0.000626
dropoff_lng         0.009446
age                -0.000554
avg_rating_given   -0.003537
churn_prob          1.000000
time_on_app        -0.001502
pages_visited      -0.003730
converted          -0.003654
rating             -0.002005
acceptance_rate    -0.002387
Name: churn_prob, dtype: float64

Mutual Information for Feature Selection

In [5]:
# We first need to change the churn prob into binary classification values
merged_DF['churn'] = (merged_DF['churn_prob']>0.5).astype(int)



In [6]:
merged_DF['churn'].value_counts()

churn
0    896128
1    105146
Name: count, dtype: int64

In [7]:
pd.set_option("display.max_columns", 40)
merged_DF

,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city_x,loyalty_status_x,signup_date_driver,loyalty_status_y,age,city_y,avg_rating_given,churn_prob,referred_by,session_id,session_time,time_on_app,pages_visited,converted,city_driver,loyalty_status,rating,vehicle_type,signup_date_rider,last_active,city_rider,acceptance_rate,churn
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze,2023-09-20,Bronze,37.629694,Nairobi,4.3,0.294348,NaN,S012312,2025-04-27 17:06:11+02:27,5.0,1.0,0.0,Nairobi,Bronze,4.1,Sedan,2024-07-21,2025-03-21 22:47:26.558938,Nairobi,0.549628,0
1,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze,2023-09-20,Bronze,37.629694,Nairobi,4.3,0.294348,NaN,S028389,2025-04-27 22:19:10+02:27,23.0,1.0,0.0,Nairobi,Bronze,4.1,Sedan,2024-07-21,2025-03-21 22:47:26.558938,Nairobi,0.549628,0
2,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze,2023-09-20,Bronze,37.629694,Nairobi,4.3,0.294348,NaN,S034943,2025-04-27 00:07:25+02:27,14.0,1.0,0.0,Nairobi,Bronze,4.1,Sedan,2024-07-21,2025-03-21 22:47:26.558938,Nairobi,0.549628,0
3,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze,2023-09-20,Bronze,37.629694,Nairobi,4.3,0.294348,NaN,S041517,2025-04-27 08:48:53+02:27,43.0,1.0,0.0,Nairobi,Bronze,4.1,Sedan,2024-07-21,2025-03-21 22:47:26.558938,Nairobi,0.549628,0
4,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 23:13:48+00:14,2024-10-28 23:26:48+00:14,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold,2024-08-30,Gold,37.051088,Lagos,4.7,0.105990,R02955,S010404,2025-04-27 01:05:16+00:14,2.0,5.0,0.0,Lagos,Gold,4.9,SUV,2023-05-06,2025-04-12 08:36:21.207528,Lagos,0.629250,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1001269,T199998,R02867,D00974,17.18,1.3,0.00,Mobile Money,2024-09-25 03:11:33+00:14,2024-09-25 03:45:33+00:14,6.540074,3.471683,6.540339,3.426481,Rainy,Lagos,Gold,2023-07-09,Gold,24.645131,Lagos,4.8,0.251196,R00665,S048082,2025-04-27 08:57:02+00:14,9.0,4.0,0.0,Lagos,Gold,3.5,Sedan,2023-09-26,2025-03-28 06:20:43.379153,Lagos,0.579188,0
1001270,T199999,R07749,D04894,13.47,1.0,0.00,Card,2024-05-24 18:19:39+02:05,2024-05-24 18:51:39+02:05,30.234277,30.884004,30.279227,30.874865,Sunny,Cairo,Bronze,2024-08-14,Bronze,34.433235,Cairo,4.1,0.324269,R07740,S004848,2025-04-27 08:02:15+02:05,10.0,1.0,0.0,Cairo,Bronze,5.0,Luxury,2023-03-30,2025-04-04 14:52:39.161225,Cairo,0.789828,0
1001271,T199999,R07749,D04894,13.47,1.0,0.00,Card,2024-05-24 18:19:39+02:05,2024-05-24 18:51:39+02:05,30.234277,30.884004,30.279227,30.874865,Sunny,Cairo,Bronze,2024-08-14,Bronze,34.433235,Cairo,4.1,0.324269,R07740,S010350,2025-04-27 07:10:56+02:05,46.0,2.0,0.0,Cairo,Bronze,5.0,Luxury,2023-03-30,2025-04-04 14:52:39.161225,Cairo,0.789828,0
1001272,T199999,R07749,D04894,13.47,1.0,0.00,Card,2024-05-24 18:19:39+02:05,2024-05-24 18:51:39+02:05,30.234277,30.884004,30.279227,30.874865,Sunny,Cairo,Bronze,2024-08-14,Bronze,34.433235,Cairo,4.1,0.324269,R07740,S019103,2025-04-27 10:20:53+02:05,21.0,1.0,1.0,Cairo,Bronze,5.0,Luxury,2023-03-30,2025-04-04 14:52:39.161225,Cairo,0.789828,0


In [8]:
merged_DF.info()

<class 'pandas.DataFrame'>
RangeIndex: 1001274 entries, 0 to 1001273
Data columns (total 37 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   trip_id             1001274 non-null  str           
 1   user_id             1001274 non-null  str           
 2   driver_id           1001274 non-null  str           
 3   fare                1001274 non-null  float64       
 4   surge_multiplier    1001274 non-null  float64       
 5   tip                 1001274 non-null  float64       
 6   payment_type        1001274 non-null  str           
 7   pickup_time         1001274 non-null  str           
 8   dropoff_time        1001274 non-null  str           
 9   pickup_lat          1001274 non-null  float64       
 10  pickup_lng          1001274 non-null  float64       
 11  dropoff_lat         1001274 non-null  float64       
 12  dropoff_lng         1001274 non-null  float64       
 13  weather             100

In [9]:

# STEPS
#1. get our features
#2. encode  catigorical features 
#3. have our x and y 
#4. fit into mutual mutual_info_classif
#5. get the mutual info scores (mi_scores)

import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

print("--- Step 1: Repairing Timestamp Objects ---")
# Enforce proper datetime formatting on pickup_time directly within your merged_DF
merged_DF['pickup_time'] = pd.to_datetime(merged_DF['pickup_time'], utc=True)

print("\n--- Step 2: Extracting and Aggregating Features safely by user_id ---")
# This step groups the bloated dataframe and collapses it back to the clean user level (~10,000 rows)
model_df = merged_DF.groupby('user_id').agg(
    # Core User Demographics (Take 'first' since it's identical across duplicates)
    age=('age', 'first'),
    city=('city_x', 'first'),  # Using city_x from your printed column schema
    loyalty_status=('loyalty_status_x', 'first'),
    avg_rating_given=('avg_rating_given', 'first'),
    
    # Aggregating Trip Performance Features (Collapsing all records safely)
    total_completed_trips=('trip_id', 'nunique'), # Count UNIQUE trips to stop duplication inflation
    avg_fare=('fare', 'mean'),
    avg_surge=('surge_multiplier', 'mean'),
    avg_tip=('tip', 'mean'),
    most_recent_trip=('pickup_time', 'max'),
    
    # Supply Chain Performance Features
    avg_driver_rating=('rating', 'mean'),
    vehicle_type=('vehicle_type', 'first'),
    avg_driver_acceptance=('acceptance_rate', 'mean'),
    
    # Aggregating Session Engagement Features
    total_app_sessions=('session_id', 'nunique'), # Count UNIQUE app entries
    avg_time_on_app=('time_on_app', 'mean'),
    avg_pages_visited=('pages_visited', 'mean'),
    total_conversions=('converted', 'sum')
).reset_index()

print(f"Compressed Dataset Shape: {model_df.shape} (Successfully normalized back to user granularity!)")

print("\n--- Step 3: Engineering the Ground Truth Churn Target Variable ---")
# Identify the last active point in the dataset
max_dataset_date = model_df['most_recent_trip'].max()

# Compute the date delta
model_df['days_since_last_trip'] = (max_dataset_date - model_df['most_recent_trip']).dt.days

# Handle any missing values for users who registered but never booked a ride
model_df['days_since_last_trip'] = model_df['days_since_last_trip'].fillna(999)
model_df['churn'] = (model_df['days_since_last_trip'] > 30).astype(int)

print(f"Target variable counts:\n{model_df['churn'].value_counts()}")

print("\n--- Step 4: Isolating the Analytical Input Vectors (X & y) ---")
y = model_df['churn']

# Drop indices, date objects, target leakages, and non-predictive keys
explicit_drops = ['user_id', 'most_recent_trip', 'churn', 'days_since_last_trip']
X_data = model_df.drop(columns=[col for col in explicit_drops if col in model_df.columns])

# Handle any residual null structures cleanly
X_data = X_data.fillna(0)

print("\n--- Step 5: Applying Safe Integer Label Encoding ---")
X_encoded = X_data.copy()
label_encoders = {}

# Convert remaining object categorical elements into memory-safe integers
object_cols = X_encoded.select_dtypes(include=['object']).columns
for col in object_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
    label_encoders[col] = le

print(f"Final clean evaluation matrix shape: {X_encoded.shape}")

print("\n--- Step 6: Computing Mutual Information Scores ---")
# Run scikit-learn's mutual info classifier calculation 
mi_scores = mutual_info_classif(X_encoded, y, random_state=42)

# Arrange data features into a clear, sorted priority table
mi_results = pd.DataFrame({
    'Feature': X_encoded.columns,
    'MI_Score': mi_scores
}).sort_values(by='MI_Score', ascending=False)

print("\n=== TOP FEATURE MUTUAL INFORMATION RANKINGS ===")
print(mi_results.to_string(index=False))

--- Step 1: Repairing Timestamp Objects ---

--- Step 2: Extracting and Aggregating Features safely by user_id ---
Compressed Dataset Shape: (10000, 17) (Successfully normalized back to user granularity!)

--- Step 3: Engineering the Ground Truth Churn Target Variable ---
Target variable counts:
churn
0    8114
1    1886
Name: count, dtype: int64

--- Step 4: Isolating the Analytical Input Vectors (X & y) ---

--- Step 5: Applying Safe Integer Label Encoding ---
Final clean evaluation matrix shape: (10000, 15)

--- Step 6: Computing Mutual Information Scores ---


C:\Users\gideo\AppData\Local\Temp\ipykernel_59352\230834437.py:75: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = X_encoded.select_dtypes(include=['object']).columns



=== TOP FEATURE MUTUAL INFORMATION RANKINGS ===
              Feature  MI_Score
total_completed_trips  0.011933
                 city  0.007438
      avg_time_on_app  0.006828
       loyalty_status  0.005904
    total_conversions  0.005849
    avg_pages_visited  0.002998
avg_driver_acceptance  0.002704
    avg_driver_rating  0.001097
            avg_surge  0.000763
                  age  0.000000
     avg_rating_given  0.000000
             avg_fare  0.000000
              avg_tip  0.000000
         vehicle_type  0.000000
   total_app_sessions  0.000000


### After Mutual information
We found that only 7 features are good enough to be used to trian our model. 
we need to engineer features that are high-signal and measures behavioural changes. 

The features we are going to engineer will consider:
1. Rider Tenure: How old is the account? (Early churn vs. late churn).

2. Trip Velocity: How frequently does this user take trips relative to their lifetime on the app?

3. App Burnout (Friction): How much time do they waste on the app without converting?

4. Driver Rejection Rate: Did they suffer a bad run of driver cancellations recently?


In [10]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

print("--- Step 1: Converting Text Dates into Real Time Objects ---")
# When pandas loads dates, it reads them as plain text strings. 
# We convert them to proper datetime objects so we can do chronological math (like subtracting dates).
# We use 'utc=True' to standardize the time zones across Cairo, Lagos, and Nairobi.
merged_DF['pickup_time'] = pd.to_datetime(merged_DF['pickup_time'], utc=True)
merged_DF['signup_date_rider'] = pd.to_datetime(merged_DF['signup_date_rider'], utc=True)
merged_DF['session_time'] = pd.to_datetime(merged_DF['session_time'], utc=True)

# We find the absolute latest date in our data. 
# We use this as a "snapshot date" to calculate how long ago events happened relative to the present day.
max_date = merged_DF['pickup_time'].max()

print("\n--- Step 2: Collapsing and Aggregating the Big Table by User ---")
# The original table has over 1 million rows because it multiplies every trip by every app session.
# We use .groupby('user_id') to collapse the data so that every unique customer gets exactly ONE clean row.
model_ready_df = merged_DF.groupby('user_id').agg(
    # Baseline Customer Information
    # We take 'first' because things like age or city are identical across a single user's duplicate rows.
    age=('age', 'first'),
    city=('city_x', 'first'),
    loyalty_status=('loyalty_status_x', 'first'),
    signup_date=('signup_date_rider', 'first'),
    
    # Financial and Transaction Summaries
    # 'nunique' counts only the distinct, individual trips, fixing the duplicated row inflation.
    total_completed_trips=('trip_id', 'nunique'),
    avg_fare=('fare', 'mean'),
    avg_surge=('surge_multiplier', 'mean'),
    avg_tip=('tip', 'mean'),
    most_recent_trip=('pickup_time', 'max'), # Finds the date of the user's very last ride
    
    # Service and Supply Quality Indicators
    avg_driver_rating=('rating', 'mean'),
    avg_driver_acceptance=('acceptance_rate', 'mean'),
    
    # App Engagement and Funnel Metrics
    total_app_sessions=('session_id', 'nunique'), # Counts distinct app openings
    avg_time_on_app=('time_on_app', 'mean'),
    avg_pages_visited=('pages_visited', 'mean'),
    total_conversions=('converted', 'sum') # Total number of times they successfully booked a ride from the app
).reset_index()

print("\n--- Step 3: Engineering Advanced Behavioral Features ---")

# Feature 1: Account Tenure
# Measures how many days the customer has been registered with RideWise.
# This helps the model distinguish between a brand new user and a loyal, long-term user.
model_ready_df['account_tenure_days'] = (max_date - model_ready_df['signup_date']).dt.days

# Feature 2: Trip Velocity
# Calculates the average number of trips a user takes per day since they signed up.
# This highlights the difference between frequent daily commuters and occasional casual riders.
# 'np.clip(..., 1, None)' prevents a mathematical crash (division by zero) if a user signed up today.
model_ready_df['trip_velocity'] = model_ready_df['total_completed_trips'] / np.clip(model_ready_df['account_tenure_days'], 1, None)

# Feature 3: Session Conversion Rate
# Calculates the percentage of app openings that actually turn into successful bookings.
# A very low percentage means the user is looking at the app but walking away without ordering a ride.
model_ready_df['session_conversion_rate'] = (model_ready_df['total_conversions'] / np.clip(model_ready_df['total_app_sessions'], 1, None)) * 100

# Feature 4: App Friction Index
# This measures wasted time. It multiplies the average time spent on the app by their failure-to-convert rate.
# A high score means the user is spending a long time browsing but leaving empty-handed—likely due to pricing or lack of drivers.
model_ready_df['app_friction_index'] = model_ready_df['avg_time_on_app'] * (1 - (model_ready_df['total_conversions'] / np.clip(model_ready_df['total_app_sessions'], 1, None)))

print("\n--- Step 4: Creating the Ground Truth Churn Target Variable (Y) ---")
# We calculate exactly how many days have passed since the user's last completed ride.
model_ready_df['days_since_last_trip'] = (max_date - model_ready_df['most_recent_trip']).dt.days

# If a user signed up but has taken zero trips, their 'days_since_last_trip' will show up as empty (NaN).
# We fill those empty spots with '999' days to signal to the model that they are completely inactive.
model_ready_df['days_since_last_trip'] = model_ready_df['days_since_last_trip'].fillna(999)

# Define our prediction target: If a user has been completely silent for more than 30 days, 
# we label them as Churned (1). If they have ridden within the last 30 days, they are Active (0).
model_ready_df['churn'] = (model_ready_df['days_since_last_trip'] > 30).astype(int)

print("\n--- Step 5: Separating Features (X) From the Target (y) ---")
# 'y' is our answer key—the label our machine learning model is trying to learn how to predict.
y = model_ready_df['churn']

# 'explicit_drops' contains columns that are IDs, raw dates, or columns used to create the target.
# We must drop these so the model doesn't cheat by looking at data from the future (Data Leakage).
explicit_drops = ['user_id', 'signup_date', 'most_recent_trip', 'days_since_last_trip', 'churn']
X_data = model_ready_df.drop(columns=[col for col in explicit_drops if col in model_ready_df.columns])

# If any empty cells (NaNs) were created during our mathematical joins, we fill them with 0 
# so the Scikit-Learn algorithms can process the matrix without throwing errors.
X_data = X_data.fillna(0)

print("\n--- Step 6: Turning Text Categories into Numbers (Label Encoding) ---")
# Machine learning models cannot read text strings like "Nairobi" or "Bronze". They only understand numbers.
# We make a copy of our data and use a LabelEncoder to translate text columns into numeric codes (e.g., Nairobi becomes 0, Lagos becomes 1).
X_encoded = X_data.copy()
label_encoders = {}

# We look for any column whose data type is an 'object' (text strings)
object_cols = X_encoded.select_dtypes(include=['object']).columns
for col in object_cols:
    le = LabelEncoder()
    # Train the encoder to learn the categories and transform the text into clean integers
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
    label_encoders[col] = le # Save the encoder so we can translate back to text later if needed

print("\n--- Step 7: Calculating New Mutual Information Scores ---")
# We run the Mutual Information algorithm to see how much predictive value our new features have.
mi_scores = mutual_info_classif(X_encoded, y, random_state=42)

# Pack the scores into a neat, user-friendly summary table sorted from highest score to lowest score.
mi_results = pd.DataFrame({
    'Feature': X_encoded.columns,
    'MI_Score': mi_scores
}).sort_values(by='MI_Score', ascending=False)

print("\n=== UPDATED FEATURE MUTUAL INFORMATION RANKINGS ===")
print(mi_results.to_string(index=False))

--- Step 1: Converting Text Dates into Real Time Objects ---

--- Step 2: Collapsing and Aggregating the Big Table by User ---

--- Step 3: Engineering Advanced Behavioral Features ---

--- Step 4: Creating the Ground Truth Churn Target Variable (Y) ---

--- Step 5: Separating Features (X) From the Target (y) ---

--- Step 6: Turning Text Categories into Numbers (Label Encoding) ---

--- Step 7: Calculating New Mutual Information Scores ---


C:\Users\gideo\AppData\Local\Temp\ipykernel_59352\18485966.py:103: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = X_encoded.select_dtypes(include=['object']).columns



=== UPDATED FEATURE MUTUAL INFORMATION RANKINGS ===
                Feature  MI_Score
  total_completed_trips  0.017687
          trip_velocity  0.011973
        avg_time_on_app  0.007000
         loyalty_status  0.005360
     total_app_sessions  0.004975
              avg_surge  0.004721
      avg_driver_rating  0.004520
    account_tenure_days  0.003959
      total_conversions  0.003762
  avg_driver_acceptance  0.002704
session_conversion_rate  0.001940
                   city  0.000450
                    age  0.000000
                avg_tip  0.000000
               avg_fare  0.000000
      avg_pages_visited  0.000000
     app_friction_index  0.000000
